In [1]:
import time
import numpy as np
import pandas as pd

# sklearn
from sklearn.cluster import KMeans as sklearnKMeans

# cuML-related libraries
import cudf
import cuml
import cupy as cp
from cuml.cluster import KMeans as cumlKMeans
print(cuml.__version__)

# metrics
from sklearn.metrics import adjusted_rand_score, calinski_harabasz_score

# data generating process
from dgp import generate_dataset # from dgp.py

26.04.000


In [2]:
# ---------------------------------------------------------
# Experiment Parameters
# ---------------------------------------------------------
n_samples    = 100000
n_features   = 500
n_clusters   = 5
n_replicates = 100

In [3]:
# Lists to store the metrics for each replicate
cpu_runtimes = []
gpu_runtimes = []
cpu_aris = []
gpu_aris = []

print(f"Starting Benchmark with {n_replicates} replicates...")
print(f"Dataset per replicate: {n_samples} samples, {n_features} features, {n_clusters} clusters")

# =========================================================
# WARM-UP PHASE 
# =========================================================
print("\n--- Performing GPU & CPU Warm-up ---")
# Generate a small dummy dataset specifically for warming up the CUDA context
X_warm, _ = generate_dataset(n_samples=1000, n_features=n_features, n_clusters=n_clusters, seed=0)
X_warm_cpu = X_warm.astype(np.float32)
X_warm_gpu = cp.asarray(X_warm_cpu)

# Run a dummy training on the CPU and GPU to load libraries into cache
_ = sklearnKMeans(n_clusters=n_clusters, n_init=1).fit(X_warm_cpu)
_ = cumlKMeans(n_clusters=n_clusters, n_init=1).fit(X_warm_gpu)
print("Warm-up complete. The actual benchmark will now begin.")

Starting Benchmark with 100 replicates...
Dataset per replicate: 100000 samples, 500 features, 5 clusters

--- Performing GPU & CPU Warm-up ---
Warm-up complete. The actual benchmark will now begin.


In [4]:
# =========================================================
# MAIN EXPERIMENT LOOP
# =========================================================

for i in range(n_replicates):
    print(f"\n--- Running Replicate {i + 1}/{n_replicates} ---")
    
    # Generate the synthetic dataset. 
    # Notice: random_state is incremented so each replicate uses new data
    X, y_true = generate_dataset(n_samples=n_samples, n_features=n_features, n_clusters=n_clusters, seed=i)
    
    # Convert and transfer data
    X_cpu = X.astype(np.float32)
    X_gpu = cp.asarray(X_cpu)

    # 1. Scikit-learn (CPU)
    start_time = time.time()
    sk_kmeans = sklearnKMeans(n_clusters=n_clusters, init='k-means++', n_init=1, max_iter=100, random_state=i)
    sk_labels = sk_kmeans.fit_predict(X_cpu)
    cpu_time = time.time() - start_time
    cpu_runtimes.append(cpu_time)
    cpu_aris.append(adjusted_rand_score(y_true, sk_labels))

    # 2. cuML (GPU)
    start_time = time.time()
    cu_kmeans = cumlKMeans(n_clusters=n_clusters, init='k-means++', n_init=1, max_iter=100, random_state=i)
    cu_labels = cu_kmeans.fit_predict(X_gpu)
    gpu_time = time.time() - start_time
    gpu_runtimes.append(gpu_time)
    cu_labels_cpu = cu_labels.get()
    gpu_aris.append(adjusted_rand_score(y_true, cu_labels_cpu))

    # =========================================================
    # SAVE RESULTS TO DISK (After the loop finishes)
    # =========================================================
    
    # Create a dictionary to organize the lists of metrics we collected during the loop
    results_dict = {
        # Generate a sequence of numbers from 1 to n_replicates to track the iteration
        "Replicate": list(range(1, i + 2)),
        # Add the Scikit-learn CPU execution times
        "CPU_Runtime_sec": cpu_runtimes,
        # Add the cuML GPU execution times
        "GPU_Runtime_sec": gpu_runtimes,
        # Add the Scikit-learn CPU Adjusted Rand Index scores
        "CPU_ARI": cpu_aris,
        # Add the cuML GPU Adjusted Rand Index scores
        "GPU_ARI": gpu_aris
    }
    
    # Convert the organized dictionary into a pandas DataFrame structure
    results_df = pd.DataFrame(results_dict)
    
    # Define the filename where the results will be saved
    output_filename = "python_kmeans_results.csv"
    
    # Save the DataFrame to a CSV file, setting index=False to prevent writing row numbers
    results_df.to_csv(output_filename, index=False)


--- Running Replicate 1/100 ---

--- Running Replicate 2/100 ---

--- Running Replicate 3/100 ---

--- Running Replicate 4/100 ---

--- Running Replicate 5/100 ---

--- Running Replicate 6/100 ---

--- Running Replicate 7/100 ---

--- Running Replicate 8/100 ---

--- Running Replicate 9/100 ---

--- Running Replicate 10/100 ---

--- Running Replicate 11/100 ---

--- Running Replicate 12/100 ---

--- Running Replicate 13/100 ---

--- Running Replicate 14/100 ---

--- Running Replicate 15/100 ---

--- Running Replicate 16/100 ---

--- Running Replicate 17/100 ---

--- Running Replicate 18/100 ---

--- Running Replicate 19/100 ---

--- Running Replicate 20/100 ---

--- Running Replicate 21/100 ---

--- Running Replicate 22/100 ---

--- Running Replicate 23/100 ---

--- Running Replicate 24/100 ---

--- Running Replicate 25/100 ---

--- Running Replicate 26/100 ---

--- Running Replicate 27/100 ---

--- Running Replicate 28/100 ---

--- Running Replicate 29/100 ---

--- Running Replicate 

- Averaged results:

In [5]:
# ---------------------------------------------------------
# Final Statistics & Benchmark Results Summary
# ---------------------------------------------------------
# Calculate mean and standard deviation for runtimes
cpu_runtime_mean, cpu_runtime_std = np.mean(cpu_runtimes), np.std(cpu_runtimes)
gpu_runtime_mean, gpu_runtime_std = np.mean(gpu_runtimes), np.std(gpu_runtimes)

# Calculate mean and standard deviation for ARI
cpu_ari_mean, cpu_ari_std = np.mean(cpu_aris), np.std(cpu_aris)
gpu_ari_mean, gpu_ari_std = np.mean(gpu_aris), np.std(gpu_aris)

# Calculate overall speedup based on mean runtimes
mean_speedup = cpu_runtime_mean / gpu_runtime_mean

print("\n" + "="*50)
print(f"    KMeans Benchmark Summary ({n_replicates} Replicates)    ")
print("-" * 50)

# Print Runtime Statistics (Mean ± Std)
print("Runtime (Seconds):")
print(f"  CPU (sklearn):  {cpu_runtime_mean:.4f} ± {cpu_runtime_std:.4f}")
print(f"  GPU (cuML):     {gpu_runtime_mean:.4f} ± {gpu_runtime_std:.4f}")
print(f"  Mean Speedup:   {mean_speedup:.2f}x faster")

print("-" * 50)

# Print Accuracy Statistics (Mean ± Std)
print("Accuracy (Adjusted Rand Index):")
print(f"  CPU (sklearn):  {cpu_ari_mean:.6f} ± {cpu_ari_std:.6f}")
print(f"  GPU (cuML):     {gpu_ari_mean:.6f} ± {gpu_ari_std:.6f}")
print(f"  Mean Diff:      {abs(cpu_ari_mean - gpu_ari_mean):.6e}")

print("="*50)


    KMeans Benchmark Summary (100 Replicates)    
--------------------------------------------------
Runtime (Seconds):
  CPU (sklearn):  3.7458 ± 1.7451
  GPU (cuML):     0.2118 ± 0.1270
  Mean Speedup:   17.69x faster
--------------------------------------------------
Accuracy (Adjusted Rand Index):
  CPU (sklearn):  0.767443 ± 0.153638
  GPU (cuML):     0.785218 ± 0.150270
  Mean Diff:      1.777479e-02
